In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

In [ ]:
from transformers import AutoTokenizer, LlamaForCausalLM

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
llama_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
llama_model = llama_model.to(device)

In [ ]:
token = llama_tokenizer.encode("ay")
token_tensor = torch.tensor([[token[1]]]).to(device)
token_tensor

In [ ]:
embedding = llama_model.model.embed_tokens(token_tensor)
embedding.shape

In [ ]:
from turkish_tokenizer import core as tt

tokens, ids = tt.tokenize("Merhaba dünya")
print(tokens)  # ['merhaba', '<space>', 'dünya']
print(ids)     # [2036, 1, 224]

In [ ]:
vocab_dict = {**tt.bpe_tokens, **tt.suffixes, **tt.roots}

vocab_size = 32768
vocab_size

In [ ]:
vocab_dict["<uppercase>"]

In [8]:
from llama3_2.config import LlamaConfig
teacher_config = LlamaConfig()


In [ ]:
from llama3_2.model_trfmrs import LlamaModel


teacher_model = LlamaModel(teacher_config)
teacher_model = teacher_model.to(device)
teacher_model

In [ ]:
teacher_model.load_state_dict(llama_model.model.state_dict())

In [11]:
def count_parameters(model):
    """Count the number of parameters in a model"""
    total_params = sum(p.numel() for p in model.parameters())
    return total_params

In [ ]:
# Count parameters for the llama_model
total_params = count_parameters(llama_model.model)
print(f"Total parameters in llama_model: {total_params:,}")

In [ ]:
total_params = count_parameters(teacher_model)
print(f"Total parameters in teacher_model: {total_params:,}")

In [ ]:
llama_model_out = llama_model.model(token_tensor)
llama_model_out

In [ ]:
teacher_model_output = teacher_model(token_tensor)
teacher_model_output

## Map Embeddings

In [16]:
def tr_capitalize(word):
  # i -> İ
  if word[0] == "i":
    return "İ" + word[1:]
  else:
    return word.capitalize()

In [ ]:
t1 = [1, 2, 3]
t2 = [4, 5, 6]

t = [t1, t2]
t = torch.tensor(t, dtype=torch.float32)

t = torch.mean(t, dim=0)
t

In [18]:
import torch.nn.functional as F


def get_vector_for_token(token, model, tokenizer):
  token_id = 0
  if token == "<uppercase>":
    token_id = 128002
  elif token == "<space>":
    token_id = 128003
  elif token == "<newline>":
    token_id = 128011
  elif token == "<tab>":
    token_id = 128012
  elif token == "<unknown>":
    token_id = 128013
  elif token.startswith("kok_") or token.startswith("ek_") or token.startswith("special_"):
    token_id = 128014
  
  if token_id > 0:
    return model.embed_tokens(torch.tensor([token_id]).to(device))

  token_ids = tokenizer.encode(token)
  token_ids0 = tokenizer.encode(tr_capitalize(token))

  if (len(token_ids)) > (len(token_ids0)):
    token_ids0.remove(128000)
    # mean of token_ids0
    token_ids0 = torch.tensor(token_ids0).to(device)
    token_vectors = model.embed_tokens(token_ids0)
    return torch.mean(token_vectors, dim=0)
  else:
    token_ids.remove(128000)
    # mean of token_ids
    token_ids = torch.tensor(token_ids).to(device)
    token_vectors = model.embed_tokens(token_ids)
    return torch.mean(token_vectors, dim=0).to(device)

  

In [ ]:
embeddings = teacher_model.embed_tokens.weight[:vocab_size].clone()
embeddings.shape

In [ ]:
embeddings_torch = torch.nn.Embedding(10, 10, device=device)
embeddings_torch

In [ ]:
embeddings_torch(torch.tensor(32790).to(device))

In [ ]:
student_config = LlamaConfig(vocab_size=vocab_size)
student_model = LlamaModel(student_config)
student_model = student_model.to(device)
student_model.layers.load_state_dict(teacher_model.layers.state_dict())
student_model.norm.load_state_dict(teacher_model.norm.state_dict())
student_model

In [ ]:
v = get_vector_for_token("a", teacher_model, llama_tokenizer)
v.shape

In [ ]:
torch.sum(v)

In [ ]:
v[3] = v[2]
v

In [ ]:
i = 3241
token = list(vocab_dict.keys())[i]
token_id = vocab_dict[token]

token, token_id

In [ ]:
""" (tensor([ 0.0045,  0.0166,  0.0210,  ..., -0.0054, -0.0422, -0.0315],
        device='mps:0'),
 tensor([ 0.0215, -0.0238,  0.0211,  ..., -0.0107, -0.0011, -0.0374],
        device='mps:0'),
 tensor([ 0.0136,  0.0104,  0.0128,  ...,  0.0081, -0.0122,  0.0051],
        device='mps:0'),
 tensor([-0.0219,  0.0057, -0.0042,  ..., -0.0008,  0.0066,  0.0013],
        device='mps:0')) """

In [ ]:
embeddings.data[0], embeddings.data[1], embeddings.data[2], embeddings.data[3]

In [29]:
embeddings.data[3] = embeddings.data[0]

In [ ]:
from tqdm import tqdm

for i in tqdm(range(len(vocab_dict.keys())), desc="Mapping embeddings to Llama embeddings"):
  token = list(vocab_dict.keys())[i]
  token_id = vocab_dict[token]
  try:
    v = get_vector_for_token(token, teacher_model, llama_tokenizer)
    embeddings.data[token_id] = v
  except Exception as e:
    print(e)
    print(token, token_id)
    break


In [37]:
student_model.embed_tokens = torch.nn.Embedding.from_pretrained(embeddings)

In [ ]:
teacher_model.eval()
student_model.eval()

In [ ]:
new_token_tensor = llama_tokenizer.encode("kok_22263")[1:]
new_token_tensor

In [ ]:
""" torch.Size([1, 1, 2048]) torch.Size([1, 1, 2048]) tensor([[ 0.0096, -0.0077, -0.0233,  ...,  0.0053, -0.0231, -0.0140]],
       device='mps:0', grad_fn=<MeanBackward1>) tensor([[-0.6481, -0.6347,  0.8181,  ...,  0.6798,  0.1396, -1.0508]],
       device='mps:0', grad_fn=<MeanBackward1>) """

In [ ]:
new_token_tensor = torch.tensor([[0]]).to(device)

teacher_model_output = teacher_model.embed_tokens(new_token_tensor)
student_model_out = student_model.embed_tokens(new_token_tensor)

print(teacher_model_output.shape, student_model_out.shape, torch.mean(teacher_model_output, dim=1), torch.mean(student_model_out, dim=1))


In [ ]:
count_parameters(teacher_model), count_parameters(student_model)

In [40]:
import safetensors


safetensors.torch.save_file(teacher_model.state_dict(), "teacher_model1.safetensors")
safetensors.torch.save_file(student_model.state_dict(), "student_model1.safetensors")